In [13]:
# Load libraries
import pandas as pd
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import scipy.stats
import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from src import drop_column_using_vif_, show_vif_values

import geopandas as gpd
from shapely.geometry import Point

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

from tqdm import tqdm
tqdm.pandas()

In [61]:
# Open Toronto Listings data
path = os.path.join('data', 'toronto_listings_data.csv')
listings = pd.read_csv(path)

# Convert to gdp dataframe
listings['geometry'] = listings.apply(lambda x: Point((x.longitude, x.latitude)), axis=1)
listings = gpd.GeoDataFrame(listings, geometry='geometry', crs='EPSG:4326')

listings = listings.drop(columns=['last_review', 'latitude', 'longitude'])

In [62]:
# Remove rows where minimum_nights is >= 28
listings = listings[listings['minimum_nights'] < 28]

In [63]:
# Open Toronto License data
path = os.path.join('data', 'toronto_license_data_geocoded.csv')
licenses = pd.read_csv(path)

# Convert to gpd dataframe
licenses['geometry'] = licenses.apply(lambda x: Point((x.longitude, x.latitude)), axis=1)
licenses = gpd.GeoDataFrame(licenses, geometry='geometry', crs='EPSG:4326')

# Drop columns latitude, longitude, coordinates, address_for_geocode
licenses = licenses.drop(columns=['latitude', 'longitude', 'coordinates', 'address_for_geocode']) 

In [64]:
# Define the regex pattern
pattern = r'^STR-\d{4}-[A-Z]{6}$'

# Apply regex matching to the "license" column
listings['valid_license_format'] = listings['license'].apply(lambda x: bool(re.match(pattern, str(x))))

In [65]:
# Whats the count of rows with valid license format?
print(listings['valid_license_format'].value_counts())  # Should return True and False counts

valid_license_format
True     6924
False     136
Name: count, dtype: int64


In [66]:
licenses

,_id,operator_registration_number,address,unit,postal_code,property_type,ward_number,ward_name,location,geometry
0,4330427,STR-2303-GZJKVV,51 Dewson St,2,M6H,Single/Semi-detached House,9,Davenport,"51, Dewson Street, Palmerston-Little Italy, Da...",POINT (-79.42562 43.65637)
1,4330428,STR-2012-FVBHHD,454 Ruth Ave,NaN,M2M,Single/Semi-detached House,18,Willowdale,"454, Ruth Avenue, Newtonbrook East, Willowdale...",POINT (-79.39734 43.79351)
2,4330429,STR-2208-GRWBHG,561 Sherbourne St,3603,M4X,Apartment,13,Toronto Centre,"561 Sherbourne, 561, Sherbourne Street, North ...",POINT (-79.37522 43.66946)
3,4330430,STR-2209-JCLRVM,91 Hove St,NaN,M3H,Single/Semi-detached House,6,York Centre,"91, Hove Street, Bathurst Manor, York Centre, ...",POINT (-79.44371 43.76088)
4,4330431,STR-2310-FYTKVR,36 Zorra St,1410,M8Z,Condominium,3,Etobicoke-Lakeshore,"36, Zorra Street, Etobicoke City Centre, Etobi...",POINT (-79.52178 43.61974)
...,...,...,...,...,...,...,...,...,...,...
8172,4338599,STR-2504-FZBPPR,61 Riverhead Dr,NaN,M9W,Single/Semi-detached House,1,Etobicoke North,"61, Riverhead Drive, Rexdale-Kipling, Etobicok...",POINT (-79.56818 43.73077)
8173,4338600,STR-2504-GXVPPV,209 Fort York Blvd,1660,M5V,Condominium,10,Spadina-Fort York,"Neptune North, 209, Fort York Boulevard, Fort ...",POINT (-79.40453 43.63706)
8174,4338601,STR-2504-HGBPPW,30 Nelson St,809,M5V,Condominium,10,Spadina-Fort York,"30, Nelson Street, Eglinton East, Scarborough ...",POINT (-79.23163 43.7465)
8175,4338602,STR-2504-HPHPPX,18 Parkview Ave,1211,M2N,Condominium,18,Willowdale,"18, Parkview Avenue, Yonge-Doris, Willowdale, ...",POINT (-79.41286 43.77117)


In [ ]:
# Convert both gdfs to epsg 32190 
licenses = licenses.to_crs(epsg=32190)
listings = listings.to_crs(epsg=32190)

In [80]:
# Join data from listings (based on license) and licenses (based on operator_registration_number)
combined_data = listings.merge(licenses, how='left', left_on='license', right_on='operator_registration_number')

In [81]:
# 2. Calculate the distance in meters
combined_data['distance_m'] = combined_data.apply(
    lambda row: row['geometry_x'].distance(row['geometry_y']) if row['geometry_x'] and row['geometry_y'] else None,
    axis=1
)

In [90]:
# 3. Create the physically_close column
combined_data['physically_close'] = combined_data['distance_m'] <= 300

In [91]:
# Count column physically_close
print(combined_data['physically_close'].value_counts())  # Should return True and False counts

physically_close
True     6191
False     869
Name: count, dtype: int64


In [92]:
# How many rows have True in both physically_close and valid_license_format?
print(combined_data[(combined_data['physically_close'] == True) & (combined_data['valid_license_format'] == True)].shape[0])  # Should return the count of rows with both conditions true

# How many rows have False in physically_close OR False in valid_license_format?
print(combined_data[(combined_data['physically_close'] == False) | (combined_data['valid_license_format'] == False)].shape[0])  # Should return the count of rows with either condition false

6191
869


In [75]:
combined_data

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,room_type,price,minimum_nights,number_of_reviews,...,address,unit,postal_code,property_type,ward_number,ward_name,location,geometry_y,distance_m,physically_close
0,696973520016945803,"Private one bedroom, two twin beds",43668850,Kunga,NaN,Alderwood,Entire home/apt,81.0,3,26,...,208 North Carson St,2,M8W,Single/Semi-detached House,3.0,Etobicoke-Lakeshore,"208, North Carson Street, Alderwood, Etobicoke...",POINT (-1072023.217 14985266.409),109.775272,True
1,697004718610327555,"Luxury 2 bedroom townhouse, with beautiful view",475771630,Gabriel G,NaN,Downsview-Roding-CFB,Entire home/apt,236.0,2,43,...,155 Downsview Park Blvd,115,M3K,Townhouse/ Row House,6.0,York Centre,"Downsview Park Boulevard, Downsview, York Cent...",POINT (-1073481.496 14970339.912),148.737248,True
2,697013674104755196,Elegant 3-Bedroom Gem in Toronto,390536890,Md Monir,NaN,Oakridge,Entire home/apt,366.0,1,48,...,196 Clonmore Dr,Upper,M1N,Single/Semi-detached House,20.0,Scarborough Southwest,"196, Clonmore Drive, Birchcliffe-Cliffside, Sc...",POINT (-1091107.098 14971289.952),113.532113,True
3,697045300656398535,Luxury 2BD & 2Bath w Stunning Lake View Downtown,475788987,Ben,NaN,Waterfront Communities-The Island,Entire home/apt,250.0,3,73,...,15 Iceboat Ter,1706,M5V,Condominium,10.0,Spadina-Fort York,"15, Iceboat Terrace, Harbourfront-CityPlace, S...",POINT (-1082734.028 14979357.586),86.532875,True
4,697177315863595052,Cozy and Bright Retro Retreat in Upper Junction,130178952,Sarah,NaN,Junction Area,Private room,57.0,4,59,...,218 Weston Rd,NaN,M6N,Single/Semi-detached House,5.0,York South-Weston,"218, Weston Road, Junction Area, York South—We...",POINT (-1075625.785 14976756.098),123.486548,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7055,1362472935884606880,Modren 4BR Home 15 mins to Downtown Sleeps 10,650760476,Furukh Zuhra,NaN,Dorset Park,Entire home/apt,219.0,1,0,...,96 Jenkinson Way,NaN,M1P,Townhouse/ Row House,21.0,Scarborough Centre,"96, Jenkinson Way, Dorset Park, Scarborough Ce...",POINT (-1090002.605 14964831.651),114.359817,True
7056,1362596373024307170,Lux House with 7 Bedrooms/3 Baths + Yoga/Sun Room,591586181,Denis,NaN,Bathurst Manor,Entire home/apt,442.0,3,0,...,101 Combe Ave,NaN,M3H,Single/Semi-detached House,6.0,York Centre,"101, Combe Avenue, Bathurst Manor, York Centre...",POINT (-1074875.94 14967204.63),157.333609,True
7057,1362606092512348797,NEW! The Penthouse - 2100 sq.ft. - Free Parking,680389450,Anne Colette,NaN,Waterfront Communities-The Island,Entire home/apt,417.0,1,0,...,20 Blue Jays Way,2401,M5V,Condominium,10.0,Spadina-Fort York,"Blue Jays Way, Harbourfront-CityPlace, Spadina...",POINT (-1083063.607 14979157.224),263.341176,True
7058,1363007198751602252,앞이보이는방,498797815,Maree,NaN,Willowdale East,Private room,70.0,4,0,...,248 Elmwood Ave,NaN,M2N,Single/Semi-detached House,18.0,Willowdale,"248, Elmwood Avenue, East Willowdale, Willowda...",POINT (-1079179.295 14964957.514),70.567691,True


In [ ]:
# How many rows have the same operator_registration_number?
print(combined_data['operator_registration_number'].nunique())

5324


In [98]:
# We need to make sure that the license number is not being used for multiple addresses. Create a new column titled "no_multiple_address_per_license" that is True if the operator_registration_number is unique in the dataset with only a single address string (in the address column) and False otherwise.
combined_data['no_multiple_address_per_license'] = combined_data.groupby('operator_registration_number')['address'].transform(lambda x: x.nunique() == 1)

In [102]:
combined_data.groupby('operator_registration_number')['address'].transform(lambda x: x.nunique() == 1).value_counts()

address
True    6548
Name: count, dtype: int64

In [105]:
# Create a new column titled "valid_license" that is True if the valid_license_format is True, physically_close is True, and no_multiple_address_per_license is True. Otherwise, it should be False.
combined_data['valid_license'] = combined_data['valid_license_format'] & combined_data['physically_close'] & combined_data['no_multiple_address_per_license']

In [107]:
# Print all column names
print(combined_data.columns)

Index(['id', 'name', 'host_id', 'host_name', 'neighbourhood_group',
       'neighbourhood', 'room_type', 'price', 'minimum_nights',
       'number_of_reviews', 'reviews_per_month',
       'calculated_host_listings_count', 'availability_365',
       'number_of_reviews_ltm', 'license', 'geometry_x',
       'valid_license_format', '_id', 'operator_registration_number',
       'address', 'unit', 'postal_code', 'property_type', 'ward_number',
       'ward_name', 'location', 'geometry_y', 'distance_m', 'physically_close',
       'no_multiple_address_per_license', 'valid_license'],
      dtype='object')


In [109]:
# Remove all rows except 'id', 'name', 'host_id', 'host_name', 'neighbourhood_group',
    #    'neighbourhood', 'room_type', 'price', 'minimum_nights',
    #    'number_of_reviews', 'reviews_per_month',
    #    'calculated_host_listings_count', 'availability_365',
    #    'number_of_reviews_ltm', 'license', 'geometry_x', 'valid_license'
combined_data = combined_data[['id', 'name', 'host_id', 'host_name', 'neighbourhood_group',
                            'neighbourhood', 'room_type', 'price', 'minimum_nights',
                            'number_of_reviews', 'reviews_per_month',
                            'calculated_host_listings_count', 'availability_365',
                            'number_of_reviews_ltm', 'license', 'geometry_x', 'valid_license']]
    
# Rename the geometry_x column to geometry
combined_data = combined_data.rename(columns={'geometry_x': 'geometry'})

In [108]:
print(listings.columns)

Index(['id', 'name', 'host_id', 'host_name', 'neighbourhood_group',
       'neighbourhood', 'room_type', 'price', 'minimum_nights',
       'number_of_reviews', 'reviews_per_month',
       'calculated_host_listings_count', 'availability_365',
       'number_of_reviews_ltm', 'license', 'geometry', 'valid_license_format'],
      dtype='object')


In [110]:
combined_data

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,geometry,valid_license
0,696973520016945803,"Private one bedroom, two twin beds",43668850,Kunga,NaN,Alderwood,Entire home/apt,81.0,3,26,0.95,1,13,10,STR-2208-JCCBHM,POINT (301583.356 4830242.21),True
1,697004718610327555,"Luxury 2 bedroom townhouse, with beautiful view",475771630,Gabriel G,NaN,Downsview-Roding-CFB,Entire home/apt,236.0,2,43,1.93,1,339,15,STR-2210-HFKKVJ,POINT (306354.453 4843975.676),True
2,697013674104755196,Elegant 3-Bedroom Gem in Toronto,390536890,Md Monir,NaN,Oakridge,Entire home/apt,366.0,1,48,1.61,1,166,26,STR-2309-HWMKVY,POINT (322811.132 4839164.848),True
3,697045300656398535,Luxury 2BD & 2Bath w Stunning Lake View Downtown,475788987,Ben,NaN,Waterfront Communities-The Island,Entire home/apt,250.0,3,73,2.45,3,203,36,STR-2209-FRQDVC,POINT (313055.405 4833357.397),True
4,697177315863595052,Cozy and Bright Retro Retreat in Upper Junction,130178952,Sarah,NaN,Junction Area,Private room,57.0,4,59,1.92,4,363,25,STR-2011-HMDRVK,POINT (307113.345 4837409.635),True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7055,1362472935884606880,Modren 4BR Home 15 mins to Downtown Sleeps 10,650760476,Furukh Zuhra,NaN,Dorset Park,Entire home/apt,219.0,1,0,NaN,2,334,0,STR-2411-GCKYBS,POINT (323370.586 4845509.499),True
7056,1362596373024307170,Lux House with 7 Bedrooms/3 Baths + Yoga/Sun Room,591586181,Denis,NaN,Bathurst Manor,Entire home/apt,442.0,3,0,NaN,1,263,0,STR-2405-GMXKVT,POINT (308615.951 4846544.609),True
7057,1362606092512348797,NEW! The Penthouse - 2100 sq.ft. - Free Parking,680389450,Anne Colette,NaN,Waterfront Communities-The Island,Entire home/apt,417.0,1,0,NaN,1,354,0,STR-2502-GLVPPT,POINT (313599.409 4833743.343),True
7058,1363007198751602252,앞이보이는방,498797815,Maree,NaN,Willowdale East,Private room,70.0,4,0,NaN,6,364,0,STR-2501-HYRYBY,POINT (313162.322 4847767.49),True


In [111]:
# Load csv
path = os.path.join('data', 'toronto_listings_complete.csv')
listings_complete = pd.read_csv(path)

In [113]:
# Remove rows where minimum_nights is >= 28
listings_complete = listings_complete[listings_complete['minimum_nights'] < 28]

In [117]:
listings_complete

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
6,696973520016945803,https://www.airbnb.com/rooms/696973520016945803,20250302144713,2025-03-04,city scrape,"Private one bedroom, two twin beds","Newly renovated, main floor suite with private...",Walk To Parks Or Shopping At Sherway Gardens M...,https://a0.muscache.com/pictures/hosting/Hosti...,43668850,...,4.77,4.62,4.62,STR-2208-JCCBHM,f,1,1,0,0,0.95
7,697004718610327555,https://www.airbnb.com/rooms/697004718610327555,20250302144713,2025-03-04,city scrape,"Luxury 2 bedroom townhouse, with beautiful view",Your family will be close to everything when y...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,475771630,...,4.70,4.86,4.63,STR-2210-HFKKVJ,f,1,1,0,0,1.93
8,697013674104755196,https://www.airbnb.com/rooms/697013674104755196,20250302144713,2025-03-04,city scrape,Elegant 3-Bedroom Gem in Toronto,Two Storey Home With Beautiful Luscious Back Y...,NaN,https://a0.muscache.com/pictures/4675255b-41c3...,390536890,...,4.73,4.69,4.65,STR-2309-HWMKVY,f,1,1,0,0,1.61
20,697045300656398535,https://www.airbnb.com/rooms/697045300656398535,20250302144713,2025-03-03,city scrape,Luxury 2BD & 2Bath w Stunning Lake View Downtown,"Lux 2 bed 2 bath condo condo, spacious open pl...","You will love the spectacular neighborhood, wi...",https://a0.muscache.com/pictures/miso/Hosting-...,475788987,...,4.96,4.96,4.75,STR-2209-FRQDVC,t,3,1,2,0,2.45
23,697177315863595052,https://www.airbnb.com/rooms/697177315863595052,20250302144713,2025-03-03,city scrape,Cozy and Bright Retro Retreat in Upper Junction,Small / Cozy bdrm w/ fun retro details located...,My home is located in the Stockyards or Upper ...,https://a0.muscache.com/pictures/miso/Hosting-...,130178952,...,5.00,4.64,4.86,STR-2011-HMDRVK,f,4,0,4,0,1.92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21647,1366350415955423954,https://www.airbnb.com/rooms/1366350415955423954,20250302144713,2025-03-03,city scrape,Happy,Enjoy a stylish stay in this special place.,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,498797815,...,NaN,NaN,NaN,STR-2501-HYRYBY,t,6,0,6,0,NaN
21650,1366729981239127587,https://www.airbnb.com/rooms/1366729981239127587,20250302144713,2025-03-02,city scrape,Gorgeous room with window view,"Relax in this calm, stylish room in a brand-ne...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,599135394,...,NaN,NaN,NaN,STR-2411-HYGFBY,f,2,0,2,0,NaN
21653,1366807096527860840,https://www.airbnb.com/rooms/1366807096527860840,20250302144713,2025-03-03,city scrape,Private 2BR2bth 15min to Airport+Free parking,Welcome to our recently renovated spacious hom...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,677383966,...,NaN,NaN,NaN,STR-2501-GYQSBV,t,3,2,1,0,NaN
21656,1366970333908472070,https://www.airbnb.com/rooms/1366970333908472070,20250302144713,2025-03-03,city scrape,The Perfect Downtown Stay,Experience the best of downtown Toronto in thi...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,681630551,...,NaN,NaN,NaN,STR-2307-GMBPHT,f,1,1,0,0,NaN


In [118]:
# Add in the column valid_license from combined_data to listings_complete based on the id column
listings_complete = listings_complete.merge(combined_data[['id', 'valid_license']], how='left', on='id')

In [120]:
# Save to csv
path = os.path.join('data', 'toronto_listings_complete.csv')
listings_complete.to_csv(path, index=False)

In [119]:
listings_complete

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,valid_license
0,696973520016945803,https://www.airbnb.com/rooms/696973520016945803,20250302144713,2025-03-04,city scrape,"Private one bedroom, two twin beds","Newly renovated, main floor suite with private...",Walk To Parks Or Shopping At Sherway Gardens M...,https://a0.muscache.com/pictures/hosting/Hosti...,43668850,...,4.62,4.62,STR-2208-JCCBHM,f,1,1,0,0,0.95,True
1,697004718610327555,https://www.airbnb.com/rooms/697004718610327555,20250302144713,2025-03-04,city scrape,"Luxury 2 bedroom townhouse, with beautiful view",Your family will be close to everything when y...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,475771630,...,4.86,4.63,STR-2210-HFKKVJ,f,1,1,0,0,1.93,True
2,697013674104755196,https://www.airbnb.com/rooms/697013674104755196,20250302144713,2025-03-04,city scrape,Elegant 3-Bedroom Gem in Toronto,Two Storey Home With Beautiful Luscious Back Y...,NaN,https://a0.muscache.com/pictures/4675255b-41c3...,390536890,...,4.69,4.65,STR-2309-HWMKVY,f,1,1,0,0,1.61,True
3,697045300656398535,https://www.airbnb.com/rooms/697045300656398535,20250302144713,2025-03-03,city scrape,Luxury 2BD & 2Bath w Stunning Lake View Downtown,"Lux 2 bed 2 bath condo condo, spacious open pl...","You will love the spectacular neighborhood, wi...",https://a0.muscache.com/pictures/miso/Hosting-...,475788987,...,4.96,4.75,STR-2209-FRQDVC,t,3,1,2,0,2.45,True
4,697177315863595052,https://www.airbnb.com/rooms/697177315863595052,20250302144713,2025-03-03,city scrape,Cozy and Bright Retro Retreat in Upper Junction,Small / Cozy bdrm w/ fun retro details located...,My home is located in the Stockyards or Upper ...,https://a0.muscache.com/pictures/miso/Hosting-...,130178952,...,4.64,4.86,STR-2011-HMDRVK,f,4,0,4,0,1.92,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7055,1366350415955423954,https://www.airbnb.com/rooms/1366350415955423954,20250302144713,2025-03-03,city scrape,Happy,Enjoy a stylish stay in this special place.,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,498797815,...,NaN,NaN,STR-2501-HYRYBY,t,6,0,6,0,NaN,True
7056,1366729981239127587,https://www.airbnb.com/rooms/1366729981239127587,20250302144713,2025-03-02,city scrape,Gorgeous room with window view,"Relax in this calm, stylish room in a brand-ne...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,599135394,...,NaN,NaN,STR-2411-HYGFBY,f,2,0,2,0,NaN,True
7057,1366807096527860840,https://www.airbnb.com/rooms/1366807096527860840,20250302144713,2025-03-03,city scrape,Private 2BR2bth 15min to Airport+Free parking,Welcome to our recently renovated spacious hom...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,677383966,...,NaN,NaN,STR-2501-GYQSBV,t,3,2,1,0,NaN,True
7058,1366970333908472070,https://www.airbnb.com/rooms/1366970333908472070,20250302144713,2025-03-03,city scrape,The Perfect Downtown Stay,Experience the best of downtown Toronto in thi...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,681630551,...,NaN,NaN,STR-2307-GMBPHT,f,1,1,0,0,NaN,True
